# Random Forest
## Random FOrest Classifer

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,classification_report,confusion_matrix
from sklearn.tree import plot_tree

In [11]:
titanic=sns.load_dataset("titanic")
titanic.head()
features=["pclass","sex","fare","embarked","age"]
target=["survived"]

# Missing data
from sklearn.impute import SimpleImputer

imp_mean=SimpleImputer(strategy="mean")
titanic[["age"]]= imp_mean.fit_transform(titanic[["age"]])
imp_freq=SimpleImputer(strategy="most_frequent")
titanic[["embarked"]]=imp_freq.fit_transform(titanic[["embarked"]])

# Encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
titanic["sex"]=le.fit_transform(titanic["sex"])
titanic["embarked"]=le.fit_transform(titanic["embarked"])

X=titanic[features]
y=titanic["survived"]


X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=42)

In [12]:
from sklearn.tree import DecisionTreeClassifier

model=DecisionTreeClassifier()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
y_pred_train=model.predict(X_train)

from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,classification_report,confusion_matrix
print("accuracy_score",accuracy_score(y_train,y_pred_train))
print("accuracy_score",accuracy_score(y_test,y_pred))
# this is classic case of overfitting




accuracy_score 0.9791332263242376
accuracy_score 0.753731343283582


In [13]:
# from sklearn.tree import plot_tree

# plt.figure(figsize=(18,10))
# plot_tree(
#     model,
#     feature_names=X.columns,
#     class_names=["Died","Survived"],
#     filled=True
# )

## Apply Random Forest Classifer

In [14]:
from sklearn.ensemble import RandomForestClassifier

rfc=RandomForestClassifier(n_estimators=201,bootstrap=True,oob_score=True,random_state=42,max_depth=4)
model=rfc.fit(X_train,y_train)
y_pred=model.predict(X_test)

print("OOB Score: ",model.oob_score_)
print("Testing Accuracy: ",accuracy_score(y_test,y_pred))

OOB Score:  0.826645264847512
Testing Accuracy:  0.8171641791044776


## Bagging Classifier

In [15]:
from sklearn.ensemble import BaggingClassifier

basemodel=DecisionTreeClassifier()
bagging=BaggingClassifier(basemodel,n_estimators=201,oob_score=True)
bagging.fit(X_train,y_train)
y_pred=bagging .predict(X_test)

print("OOB Score: ",bagging.oob_score_)
print("Testing Accuracy: ",accuracy_score(y_test,y_pred))

OOB Score:  0.797752808988764
Testing Accuracy:  0.7798507462686567


## Apply Random Forest Regessor

In [16]:
from sklearn.datasets import load_diabetes
df=load_diabetes(as_frame=True).frame
df.head()
X=df.drop(columns= "target",axis=1)
y=df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [17]:
rfr=RandomForestRegressor(n_estimators=201,bootstrap=True,random_state=42,oob_score=True,max_depth=15,max_features='sqrt')
model=rfr.fit(X_train,y_train)
y_pred_test=model.predict(X_test)


In [18]:
from sklearn.metrics import root_mean_squared_error,r2_score,mean_absolute_error,mean_squared_error,accuracy_score
y_pred_test=model.predict(X_test)
y_pred_train=model.predict(X_train)

print("OOB Score: ",model.oob_score_)


print("\n")

print("root_mean_squared_error Train",root_mean_squared_error(y_train,y_pred_train))
print("root_mean_squared_error test",root_mean_squared_error(y_test,y_pred_test))

print("mean_squared_error Train",mean_squared_error(y_train,y_pred_train))
print("mean_squared_error Test",mean_squared_error(y_test,y_pred_test))

print("r2_score train",r2_score(y_train,y_pred_train))
print("r2_score test",r2_score(y_test,y_pred_test))

OOB Score:  0.44570905186009535


root_mean_squared_error Train 21.372145237449956
root_mean_squared_error test 54.01949431053442
mean_squared_error Train 456.7685920506549
mean_squared_error Test 2918.1057655658606
r2_score train 0.9248290531537616
r2_score test 0.44922193548290046


In [19]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score, mean_squared_error

# 1. Load data
df = load_diabetes(as_frame=True).frame
X = df.drop(columns="target")
y = df["target"]

# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Baseline: Linear Regression (to see if Random Forest actually helps)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_test_lr = lr.predict(X_test)
print("=== Baseline (Linear Regression) ===")
print(f"Test R²: {r2_score(y_test, y_pred_test_lr):.4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, y_pred_test_lr):.4f}\n")

# 4. Random Forest with hyperparameter tuning (GridSearchCV)
# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Use a smaller grid for speed; you can expand if needed
# For demonstration, we use a reduced grid:
param_grid_stricter = {
    'n_estimators': [150, 250],
    'max_depth': [3, 4, 5],           # Keep it shallow
    'min_samples_split': [10, 15, 20], # Drastically increase this!
    'min_samples_leaf': [8, 12, 16],   # Drastically increase this!
    'max_features': ['sqrt', 0.5, 0.7] # Try fractional features
}

rfr = RandomForestRegressor(
    bootstrap=True, 
    oob_score=True, 
    random_state=42,
    max_samples=0.7  # <-- ADD THIS! Use only 70% of rows per tree
)

# Perform Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=rfr,
    param_grid=param_grid_stricter,
    cv=3,
    scoring='r2',
    n_jobs=-1,          # use all cores
    verbose=1
)
grid_search.fit(X_train, y_train)

best_rfr = grid_search.best_estimator_

# 5. Evaluate on test set
y_pred_test = best_rfr.predict(X_test)
y_pred_train = best_rfr.predict(X_train)

# 6. Print results
print("=== Tuned Random Forest ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"OOB Score: {best_rfr.oob_score_:.4f}")
print("\nTraining set:")
print(f"  R²: {r2_score(y_train, y_pred_train):.4f}")
print(f"  RMSE: {root_mean_squared_error(y_train, y_pred_train):.4f}")
print(f"  MSE: {mean_squared_error(y_train, y_pred_train):.4f}")
print("\nTest set:")
print(f"  R²: {r2_score(y_test, y_pred_test):.4f}")
print(f"  RMSE: {root_mean_squared_error(y_test, y_pred_test):.4f}")
print(f"  MSE: {mean_squared_error(y_test, y_pred_test):.4f}")

# 7. (Optional) Cross-validation scores on the whole training set
cv_scores = cross_val_score(best_rfr, X_train, y_train, cv=5, scoring='r2')
print(f"\nCross-validation R² (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

=== Baseline (Linear Regression) ===
Test R²: 0.4526
Test RMSE: 53.8534

Fitting 3 folds for each of 162 candidates, totalling 486 fits
=== Tuned Random Forest ===
Best parameters: {'max_depth': 4, 'max_features': 0.5, 'min_samples_leaf': 8, 'min_samples_split': 10, 'n_estimators': 150}
OOB Score: 0.4601

Training set:
  R²: 0.5932
  RMSE: 49.7156
  MSE: 2471.6386

Test set:
  R²: 0.4861
  RMSE: 52.1791
  MSE: 2722.6543

Cross-validation R² (mean ± std): 0.4343 ± 0.1009
